# API Calling to ChatGPT

### Step 1: Setting Up OpenAI API Access

In [ ]:
!pip install openai pydantic
!pip install datasets

In [27]:
import os
from dotenv import load_dotenv
from openai import OpenAI  
from pydantic import BaseModel
from typing import List
load_dotenv() # load the .env file to get the environment variables

# Initialize client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### Step 2: Preparing the Dataset

In [28]:
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
 
# Load dataset from HuggingFace
print("Loading product dataset from Huggingface")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
 # Create images directory
IMAGES_DIR.mkdir(exist_ok=True)
print(f"\n Saving images to: {IMAGES_DIR}")

for idx, row in enumerate(dataset):
    try:
        image = row["image"]  # PIL Image object from HuggingFace
        image_path = IMAGES_DIR / f"product_{idx}.jpg"
        image.save(image_path)
    except Exception as e:
        print(f"⚠️  Could not save image {idx}: {e}")
        continue

print(f"\n✅ Dataset prepared!")
print(f"   Total products : {len(products_df)}")
print(f"   Images saved to: {IMAGES_DIR}")
 

Loading product dataset from Huggingface
✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

 Saving images to: product_images

✅ Dataset prepared!
   Total products : 100
   Images saved to: product_images


Checking the product information:

In [29]:
products_df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,<PIL.JpegImagePlugin.JpegImageFile image mode=...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,<PIL.JpegImagePlugin.JpegImageFile image mode=...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,<PIL.Image.Image image mode=L size=60x80 at 0x...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants,<PIL.JpegImagePlugin.JpegImageFile image mode=...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,<PIL.Image.Image image mode=RGB size=60x80 at ...


### Step 3: Encoding Images for API

In [31]:
import os
import base64
import pandas as pd
from pathlib import Path
from PIL import Image
from io import BytesIO

# 1. Configuration 
IMAGES_DIR = Path("product_images")

# 2. Base64 Encoding Function 
def image_to_base64(image_path: Path) -> str: #read JPG file and convert to base64 string
    with open(image_path, "rb") as f:  # Read as binary
        return base64.b64encode(f.read()).decode("utf-8")


#  3. Encode All Images from Disk
print(f" Reading images from: {IMAGES_DIR}")

image_paths = sorted(IMAGES_DIR.glob("*.jpg"))  # Get all JPGs
print(f"   Found {len(image_paths)} images")

encoded_images = []

for image_path in image_paths:
    try:
        b64_string = image_to_base64(image_path)

        encoded_images.append({
            "image_name" : image_path.name,
            "image_path" : str(image_path),
            "base64"     : b64_string,
            "b64_length" : len(b64_string)
        })

    except Exception as e:
        print(f"⚠️  Could not process {image_path.name}: {e}")
        continue

print(f"✅ Encoded {len(encoded_images)} images successfully")

# 4. Store in DataFrame 
encoded_df = pd.DataFrame(encoded_images)
print(encoded_df[["image_name", "b64_length"]].head())  # Preview

# 5. Checkpoint — Verify Encoding 
print("\n CHECKPOINT: Verifying base64 encoding...")

sample = encoded_images[0]

# Check 1: String is not empty
assert len(sample["base64"]) > 0, "❌ Base64 string is empty!"
print(f"  ✅ Check 1 — Base64 string is not empty")

# Check 2: Decodes back to bytes
try:
    decoded_bytes = base64.b64decode(sample["base64"])
    print(f"  ✅ Check 2 — Decodes back to bytes: {len(decoded_bytes):,} bytes")
except Exception as e:
    print(f"  ❌ Check 2 failed: {e}")

# Check 3: Bytes reconstruct a valid image
try:
    reconstructed = Image.open(BytesIO(decoded_bytes))
    print(f"  ✅ Check 3 — Valid image: {reconstructed.mode} {reconstructed.size}")
except Exception as e:
    print(f"  ❌ Check 3 failed: {e}")

# Check 4: Preview the string
print(f"  ✅ Check 4 — Base64 preview: {sample['base64'][:40]}...")

print(f"\n✅ All checks passed! {len(encoded_images)} images ready for API transmission.")

 Reading images from: product_images
   Found 100 images
✅ Encoded 100 images successfully
       image_name  b64_length
0   product_0.jpg        2388
1   product_1.jpg        2292
2  product_10.jpg        1760
3  product_11.jpg        1756
4  product_12.jpg        1596

 CHECKPOINT: Verifying base64 encoding...
  ✅ Check 1 — Base64 string is not empty
  ✅ Check 2 — Decodes back to bytes: 1,789 bytes
  ✅ Check 3 — Valid image: RGB (60, 80)
  ✅ Check 4 — Base64 preview: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...

✅ All checks passed! 100 images ready for API transmission.


### Step 4: Creating the Product Listing Prompt

In [32]:
def create_product_listing_prompt(
    product_name: str,
    price: float,
    category: str,
    additional_info: str = None
) -> str:
    """
    Create a prompt for generating product listings.

    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information

    Returns:
    - Formatted prompt string
    """

    #  Build optional fields cleanly 
    optional_fields = ""
    if additional_info:
        optional_fields += f"- Additional Info   : {additional_info}\n"
    
    prompt = f"""You are an expert e-commerce copywriter specializing in compelling product listings.
Analyze the product image carefully and create a listing that drives conversions.

Product Information:
- Name     : {product_name}
- Price    : ${price:.2f}
- Category : {category}
{optional_fields}
Instructions:
1. **Product Title**
   - Catchy and SEO-friendly
   - Maximum 60 characters
   - Include the most important keyword naturally

2. **Product Description** (150-200 words)
   - Start with the strongest benefit, not a feature
   - Mention colors, materials, and design elements visible in the image
   - Use persuasive, active language
   - End with a subtle call to action

3. **Key Features** (5-7 bullet points)
   - Lead each point with a benefit, not just a feature
   - Example: "All-day comfort — ergonomic design reduces fatigue"

4. **SEO Keywords** (10-15 keywords, comma-separated)
   - Mix short-tail and long-tail keywords
   - Include category, use-case, and audience terms

Important Rules:
- Be specific about what you see in the image
- Never invent features not visible or stated
- Keep the tone consistent with the {category} category
- Respond ONLY with valid JSON — no extra text or markdown

Format your response as JSON:
{{
    "title"       : "Product title here",
    "description" : "Full description here",
    "features"    : ["Benefit — Feature detail", ...],
    "keywords"    : "keyword1, keyword2, ..."
}}"""

    return prompt


#  Test prompt creation 
test_prompt = create_product_listing_prompt(
    product_name     = "Wireless Bluetooth Headphones",
    price            = 79.99,
    category         = "Electronics",
    additional_info  = "Noise cancelling, 30-hour battery"
    
)

print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...") #show first 500 characters

#  Validation checkpoint 
print("\n CHECKPOINT: Validating prompt...")
assert "{product_name}" not in test_prompt, "❌ product_name not injected"
assert "$79.99"          in test_prompt,    "❌ price not formatted correctly"
assert "Electronics"     in test_prompt,    "❌ category not injected"
print("  ✅ All variables injected correctly")
print(f"  ✅ Prompt length: {len(test_prompt):,} characters")



PROMPT TEMPLATE
You are an expert e-commerce copywriter specializing in compelling product listings.
Analyze the product image carefully and create a listing that drives conversions.

Product Information:
- Name     : Wireless Bluetooth Headphones
- Price    : $79.99
- Category : Electronics
- Additional Info   : Noise cancelling, 30-hour battery

Instructions:
1. **Product Title**
   - Catchy and SEO-friendly
   - Maximum 60 characters
   - Include the most important keyword naturally

2. **Product Description...

 CHECKPOINT: Validating prompt...
  ✅ All variables injected correctly
  ✅ Prompt length: 1,482 characters


### Step 5: Calling the ChatGPT API with Vision

Note: the following code needs to be improved for given cases when ChatGPT refuses to process an image. For now, I have just applied a quick fix, as there were problems with the first image, so I started with the second one.

In [41]:
import os
import json
import base64
import pandas as pd
from pathlib import Path
from PIL import Image
from io import BytesIO
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# 1. Vision API Call Function 
def analyze_product_image(
    image_path: str,
    product_name: str,
    price: float,
    category: str,
    additional_info: str = None,
    model: str = "gpt-4o"
) -> dict:
    """
    Send image + prompt to ChatGPT Vision API and return parsed JSON.

    Args:
        image_path   : Path to the JPG file on disk
        product_name : Name of the product
        price          : Product price
        category       : Product category
        additional_info: Optional extra details
        model          : OpenAI model to use (default: gpt-4o)

    Returns:
        Parsed JSON dict with title, description, features, keywords
    """
    
    # Step 1 — Encode image to base64
    with open(image_path, "rb") as f:
        b64_image = base64.b64encode(f.read()).decode("utf-8")

    # Step 2 — Build the prompt
    prompt = create_product_listing_prompt(
        product_name    = product_name,
        price           = price,
        category        = category,
        additional_info = additional_info
    )

    # Step 3 — Build the API request with image + text
    response = client.chat.completions.create(
        model      = model,
        max_tokens = 1000,
        messages   = [
            {
                "role": "user",
                "content": [
                    {
                        # Text prompt
                        "type": "text",
                        "text": prompt
                    },
                    {
                        # Base64 image
                        "type": "image_url",
                        "image_url": {
                            "url"   : f"data:image/jpeg;base64,{b64_image}",
                            "detail": "high"  # "high" = more detail, "low" = faster & cheaper
                        }
                    }
                ]
            }
        ]
    )

    # Step 4 — Extract raw text response
    raw_response = response.choices[0].message.content
    print(f"    Raw response received ({len(raw_response)} characters)")

    # Step 5 — Parse JSON safely
    try:
        # Clean potential markdown fences just in case
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]

        parsed = json.loads(cleaned)
        return parsed

    except json.JSONDecodeError as e:
        print(f"   ⚠️  JSON parsing failed: {e}")
        print(f"   Raw response: {raw_response[:200]}...")
        return {"error": "JSON parsing failed", "raw": raw_response}


# 2. Test on a Single Image 
IMAGES_DIR = Path("product_images")
image_paths = sorted(list(IMAGES_DIR.glob("*.jpg")))

print("="*50)
print(" CALLING CHATGPT VISION API")
print("="*50)

test_image = image_paths[1]  # Use first image as test. Testing with second image as there seems to be a problem with the first one.
print(f"\n Testing with: {test_image.name}")

result = analyze_product_image(
    image_path      = test_image,
    product_name    = "Fashion Product",
    price           = 29.99,
    category        = "Fashion",
    additional_info = "From HuggingFace fashion dataset"
)

# 3. Checkpoint — Validate Response 
print("\n CHECKPOINT: Validating API response...")

expected_keys = ["title", "description", "features", "keywords"]

for key in expected_keys:
    assert key in result, f"❌ Missing key: {key}"
    print(f"  ✅ '{key}' present")

# Validate constraints
assert len(result["title"]) <= 60,        "❌ Title exceeds 60 characters"
assert isinstance(result["features"], list), "❌ Features should be a list"
assert 5 <= len(result["features"]) <= 7, "❌ Features should have 5-7 items"

print(f"  ✅ Title length     : {len(result['title'])} characters")
print(f"  ✅ Features count   : {len(result['features'])} items")
print(f"  ✅ Description words: {len(result['description'].split())} words")

# 4. Display Result
print("\n" + "="*50)
print("GENERATED PRODUCT LISTING")
print("="*50)
print(f"\n🏷️  Title      : {result['title']}")
print(f"\n📝 Description : {result['description']}")
print(f"\n⭐ Features    :")
for f in result["features"]:
    print(f"   • {f}")
print(f"\n🔍 Keywords    : {result['keywords']}")
print(f"\n✅ API call successful! Listing ready.")

 CALLING CHATGPT VISION API

 Testing with: product_1.jpg
    Raw response received (1048 characters)

 CHECKPOINT: Validating API response...
  ✅ 'title' present
  ✅ 'description' present
  ✅ 'features' present
  ✅ 'keywords' present
  ✅ Title length     : 34 characters
  ✅ Features count   : 5 items
  ✅ Description words: 79 words

GENERATED PRODUCT LISTING

🏷️  Title      : Stylish Men's Slim Fit Denim Jeans

📝 Description : Upgrade your wardrobe with these trendy slim fit denim jeans, perfect for casual outings or a night on the town. Tailored for modern comfort, these jeans offer a sleek and stylish silhouette, accentuating your look. The deep indigo wash adds a classic touch while the durable fabric ensures lasting wear, making them a versatile addition to your fashion collection. Pair with a t-shirt for a relaxed vibe or dress it up with a blazer for a smart, polished appearance.

⭐ Features    :
   • Sleek silhouette — Slim fit design enhances style
   • Versatile style — Deep 

### Step 6: Processing Multiple Products

In [42]:
import os
import json
import time
import base64
import pandas as pd
from pathlib import Path
from PIL import Image
from io import BytesIO
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

#  1. Configuration 
IMAGES_DIR   = Path("product_images")
OUTPUT_DIR   = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

BATCH_SIZE   = 10       # Process in chunks — easy to resume if interrupted
SLEEP_TIME   = 1.5      # Seconds between API calls — avoids rate limits
MAX_RETRIES  = 3        # Retry failed images up to 3 times

image_paths  = sorted(list(IMAGES_DIR.glob("*.jpg")))
print(f"   Found {len(image_paths)} images to process")


#  2. Helper Functions 
def image_to_base64(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def create_product_listing_prompt(
    product_name: str,
    price: float,
    category: str,
    additional_info: str = None
) -> str:
    optional = f"- Additional Info: {additional_info}\n" if additional_info else ""
    return f"""You are an expert e-commerce copywriter specializing in compelling product listings.
Analyze the product image carefully and create a listing that drives conversions.

Product Information:
- Name     : {product_name}
- Price    : ${price:.2f}
- Category : {category}
{optional}
Instructions:
1. **Product Title** — Catchy, SEO-friendly, max 60 characters
2. **Product Description** (150-200 words) — Start with strongest benefit, mention visual details
3. **Key Features** (5-7 bullet points) — Benefit-led
4. **SEO Keywords** (10-15, comma-separated) — Mix short and long-tail

Important Rules:
- Be specific about what you see in the image
- Never invent features not visible or stated
- Respond ONLY with valid JSON — no extra text or markdown

{{
    "title"       : "Product title here",
    "description" : "Full description here",
    "features"    : ["Benefit — Feature detail", ...],
    "keywords"    : "keyword1, keyword2, ..."
}}"""


def parse_json_response(raw: str) -> dict:
    cleaned = raw.strip()
    if "```" in cleaned:
        parts = cleaned.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            try:
                return json.loads(part)
            except:
                continue
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    start = cleaned.find("{")
    end   = cleaned.rfind("}") + 1
    if start != -1 and end > start:
        try:
            return json.loads(cleaned[start:end])
        except:
            pass
    return {"error": "JSON parsing failed", "raw": raw}


def analyze_product_image(
    image_path: Path,
    product_name: str,
    price: float,
    category: str,
    additional_info: str = None,
    model: str = "gpt-4o"
) -> dict:
    b64_image = image_to_base64(image_path)
    prompt    = create_product_listing_prompt(
        product_name, price, category, additional_info
    )
    response  = client.chat.completions.create(
        model      = model,
        max_tokens = 1000,
        messages   = [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {
                    "url"   : f"data:image/jpeg;base64,{b64_image}",
                    "detail": "high"
                }}
            ]
        }]
    )
    return parse_json_response(response.choices[0].message.content)


#  3. Batch Processing with Retry Logic 
def process_batch(image_paths: list) -> tuple[list, list]:
    """
    Process a list of images with retry logic.
    Returns: (successful_results, failed_results)
    """
    results = []
    failed  = []

    for idx, image_path in enumerate(image_paths):
        print(f"\n  [{idx+1}/{len(image_paths)}] {image_path.name}")
        success = False

        # Retry loop
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                listing = analyze_product_image(
                    image_path      = image_path,
                    product_name    = "Fashion Product",
                    price           = 29.99,
                    category        = "Fashion",
                    additional_info = "From HuggingFace fashion dataset"
                )

                if "error" in listing:
                    raise ValueError(listing["error"])

                results.append({
                    "image_name"  : image_path.name,
                    "status"      : "success",
                    "title"       : listing.get("title", ""),
                    "description" : listing.get("description", ""),
                    "features"    : " | ".join(listing.get("features", [])),
                    "keywords"    : listing.get("keywords", ""),
                    "processed_at": datetime.now().isoformat()
                })

                print(f"    ✅ '{listing['title']}'")
                success = True
                break  # Exit retry loop on success

            except Exception as e:
                print(f"    ⚠️  Attempt {attempt}/{MAX_RETRIES} failed: {e}")
                if attempt < MAX_RETRIES:
                    time.sleep(SLEEP_TIME * attempt)  # Backoff: wait longer each retry
                else:
                    failed.append({
                        "image_name" : image_path.name,
                        "status"     : "failed",
                        "error"      : str(e),
                        "processed_at": datetime.now().isoformat()
                    })
                    print(f"    ❌ Giving up after {MAX_RETRIES} attempts")

        if success:
            time.sleep(SLEEP_TIME)  # Polite delay between successful calls

    return results, failed


#  4. Run Batch in Chunks 
all_results = []
all_failed  = []

print("\n" + "="*50)
print(f"   STARTING BATCH — {len(image_paths)} images")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Max retries: {MAX_RETRIES}")
print(f"   Sleep time : {SLEEP_TIME}s")
print("="*50)

# Split into chunks
chunks = [image_paths[i:i+BATCH_SIZE] for i in range(0, len(image_paths), BATCH_SIZE)]

for chunk_idx, chunk in enumerate(chunks):
    print(f"\n BATCH {chunk_idx+1}/{len(chunks)} ({len(chunk)} images)")
    print("-"*40)

    batch_results, batch_failed = process_batch(chunk)

    all_results.extend(batch_results)
    all_failed.extend(batch_failed)

    # Save after every batch — never lose progress
    pd.DataFrame(all_results).to_csv(OUTPUT_DIR / "product_listings.csv", index=False)
    pd.DataFrame(all_failed).to_csv(OUTPUT_DIR  / "failed_listings.csv",  index=False)
    
    # Save JSON format as well 
    with open(OUTPUT_DIR / "product_listings.json", "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\n  💾 Progress saved — {len(all_results)} done, {len(all_failed)} failed")
    print(f"  ⏳ Pausing before next batch...")
    time.sleep(SLEEP_TIME * 2)  # Extra pause between batches


#  5. Checkpoint — Verify Results 
print("\n" + "="*50)
print("🔍 CHECKPOINT: Verifying batch results...")
print("="*50)

results_df = pd.DataFrame(all_results)
failed_df  = pd.DataFrame(all_failed)

# Check 1: Multiple products processed
assert len(all_results) > 1, "❌ Less than 2 products processed"
print(f"  ✅ Check 1 — Multiple products processed : {len(all_results)}")

# Check 2: CSV saved and readable
saved_df = pd.read_csv(OUTPUT_DIR / "product_listings.csv")
assert len(saved_df) == len(all_results), "❌ CSV row count mismatch"
print(f"  ✅ Check 2 — CSV saved correctly         : {len(saved_df)} rows")

# Check 2b: JSON saved and readable
with open(OUTPUT_DIR / "product_listings.json") as f:
    saved_json = json.load(f)
assert len(saved_json) == len(all_results), "❌ JSON record count mismatch"
print(f"  ✅ Check 2b — JSON saved correctly        : {len(saved_json)} records")

# Check 3: Required columns present
required_cols = ["image_name", "title", "description", "features", "keywords"]
for col in required_cols:
    assert col in saved_df.columns, f"❌ Missing column: {col}"
print(f"  ✅ Check 3 — All required columns present: {required_cols}")

# Check 4: Errors handled gracefully
total     = len(all_results) + len(all_failed)
success_rate = (len(all_results) / total * 100) if total > 0 else 0
print(f"  ✅ Check 4 — Errors handled gracefully")
print(f"             Success : {len(all_results)}/{total} ({success_rate:.1f}%)")
print(f"             Failed  : {len(all_failed)}/{total}")


#  6. Final Summary 
print("\n" + "="*50)
print(" FINAL SUMMARY")
print("="*50)
print(f"  ✅ Processed  : {len(all_results)} products")
print(f"  ❌ Failed     : {len(all_failed)} products")
print(f"  📈 Success    : {success_rate:.1f}%")
print(f"  💾 Listings   : {OUTPUT_DIR}/product_listings.csv")
print(f"  💾 JSON       : {OUTPUT_DIR}/product_listings.json")
print(f"  💾 Failed log : {OUTPUT_DIR}/failed_listings.csv")
print(f"\n  Preview of results:")
print(results_df[["image_name", "title"]].head())

   Found 100 images to process

   STARTING BATCH — 100 images
   Batch size : 10
   Max retries: 3
   Sleep time : 1.5s

 BATCH 1/10 (10 images)
----------------------------------------

  [1/10] product_0.jpg
    ⚠️  Attempt 1/3 failed: JSON parsing failed
    ⚠️  Attempt 2/3 failed: JSON parsing failed
    ⚠️  Attempt 3/3 failed: JSON parsing failed
    ❌ Giving up after 3 attempts

  [2/10] product_1.jpg
    ✅ 'Stylish Men's Slim Fit Jeans - Blue Denim'

  [3/10] product_10.jpg
    ✅ 'Sleek Black Athletic Shoe - Ultimate Comfort'

  [4/10] product_11.jpg
    ✅ 'Stylish Black Belt with Silver Buckle'

  [5/10] product_12.jpg
    ✅ 'Stylish Comfort Flip Flops for Everyday Wear'

  [6/10] product_13.jpg
    ✅ 'Chic Abstract Print Shoulder Bag - $29.99'

  [7/10] product_14.jpg
    ⚠️  Attempt 1/3 failed: JSON parsing failed
    ⚠️  Attempt 2/3 failed: JSON parsing failed
    ⚠️  Attempt 3/3 failed: JSON parsing failed
    ❌ Giving up after 3 attempts

  [8/10] product_15.jpg
    ✅ 'El

Personal Notes:

In general, calling ChatGPT seems to work well, but the output varies. Sometimes the error is lower and sometimes it is higher. I need to investigate why that happens in the code.

Also, many times I have had to re-import libraries even though they were called at the beginning. I also need to investigate this in order to simplify the code.